# Dynamic RLS & Organisational Hierarchy Builder (`support_rls_hierarchy_gold`)

This notebook builds the Gold organisational hierarchy table (`support_rls_hierarchy_gold`) from `support_rls_dynamic_merged` with built-in safeguards to guarantee a **single connected tree** rooted at `PersonFk = 4037`:
1. **Root Integrity & Explicit Overrides:** Forces `4037` as the single root (`SupervisorId = NULL`) and applies version-controlled manual supervisor overrides.
2. **Leaver Path Compression:** Bridges active staff across departed line managers (`Leaver = 1`) to their nearest active ancestor (`HierarchyStatus = 'Bridged via Leaver'`).
3. **Token-Safe Cycle Detection:** Detects and breaks any supervisory loops without substring matching collisions.
4. **Tiered Org-Unit Fallback:** Attaches any remaining orphaned subtrees to the shallowest active supervisor in their `Level6` $\rightarrow$ `CostCentre` $\rightarrow$ `Level3` $\rightarrow$ Root (`4037`).

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from collections import deque, defaultdict

# ==============================================================================
# 1. CONFIGURATION & EXPLICIT MANUAL OVERRIDES
# ==============================================================================
ROOT_PERSON_FK = "4037"  # Managing Director (Absolute Root of SCW CSU)
SOURCE_TABLE = "support_rls_dynamic_merged"
TARGET_TABLE = "support_rls_hierarchy_gold"

# Explicit supervisor overrides previously hidden in Base64 support_rls_dynamic_missing.
MANUAL_SUPERVISOR_OVERRIDES = {
    "5961": "324",   # Michael Dunne -> Nicole Sumerhill
    "2854": "3122",  # David Rowlands -> Linday Lipett
    "6112": "2904",  # Kait Harvey -> Heather Muldoon
    "6980": "2876",  # Laura Davy -> 2876
    "3252": "4874",  # Wendy Marshall -> John O'Connell
    "6688": "6656",  # Robert Pollard -> Emma Huthart
    "6678": "8779",  # Andrew Lang -> Noloyiso Xuba
    "6695": "6686",  # Elizabeth Lawrence -> Pippa Green
    "6656": "5581",  # Emma Huthart -> Nicholas Rogers
    "6686": "8779",  # Pippa Green -> Noloyiso Xuba
    "5581": "6751",  # Nicholas Rogers -> Deborah Haith
    "6751": "6935",  # Deborah Haith -> Alex Evans
    "8178": "6751",  # Ashdown -> Deborah Haith
    "8779": "6935",  # Noloyiso Xuba -> Alex Evans
    "8752": "8404",  # Bex Guy -> Matthew Tucker
    "3207": "4754",  # Steven Aulman -> Simon Sturgeon
}

# ==============================================================================
# 2. DETERMINISTIC GRAPH RESOLUTION & SAFEGUARDS ENGINE
# ==============================================================================
def clean_id(val):
    if val is None:
        return None
    s = str(val).strip()
    if s == "" or s.lower() in ("null", "none", "nan", "0", "-1"):
        return None
    if s.endswith(".0"):
        s = s[:-2]
    return s


def resolve_staff_hierarchy(rows, root_id=ROOT_PERSON_FK, manual_overrides=None):
    """
    Resolves the staff reporting hierarchy with full safeguards:
      1. Normalizes IDs and applies explicit manual overrides & root integrity.
      2. Bridges across departed managers (Leaver == 1) via path compression so
         active staff connect to their nearest active ancestor.
      3. Detects and breaks any supervisory cycles using token-exact set membership.
      4. Repairs any remaining orphaned/disconnected subtrees via deterministic
         tiered fallback (Level6 Lead -> CostCentre Lead -> Level3 Lead -> Root 4037),
         preserving intact subtrees under their team manager.
      5. Computes OrgPath, HierarchyStatus, HierarchyDepth, DirectReportsCount,
         and TotalReportsCount for every active staff member.
    """
    if manual_overrides is None:
        manual_overrides = {}

    # Identify all PersonFks referenced as a SupervisorId by anyone in the dataset
    referenced_supervisors = set()
    for r in rows:
        sup_id = clean_id(r.get("SupervisorId"))
        if sup_id:
            referenced_supervisors.add(sup_id)
    for ov_sup in manual_overrides.values():
        c_sup = clean_id(ov_sup)
        if c_sup:
            referenced_supervisors.add(c_sup)

    # Index all records by cleaned PersonFk (preferring Leaver == 0 if duplicates exist)
    nodes = {}
    for r in rows:
        pid = clean_id(r.get("PersonFk"))
        if not pid:
            continue
        leaver_val = int(r.get("Leaver") or 0)
        if pid == root_id:
            leaver_val = 0  # Root is always active
        existing = nodes.get(pid)
        if existing is None or (existing["Leaver"] != 0 and leaver_val == 0):
            raw_sup = clean_id(r.get("SupervisorId"))
            nodes[pid] = {
                "PersonFk": pid,
                "OriginalSupervisorId": raw_sup,
                "RawSupervisorId": raw_sup,
                "Leaver": leaver_val,
                "Level3": (r.get("Level3") or "").strip(),
                "CostCentre": (r.get("CostCentre") or "").strip(),
                "Level6": (r.get("Level6") or "").strip(),
                "IsSupervisor": pid in referenced_supervisors,
            }

    # Apply manual overrides & enforce Root integrity
    overridden_pids = set()
    for pid, node in nodes.items():
        if pid == root_id:
            node["RawSupervisorId"] = None
        elif pid in manual_overrides:
            override_sup = clean_id(manual_overrides[pid])
            if override_sup and override_sup != pid:
                node["RawSupervisorId"] = override_sup
                overridden_pids.add(pid)
        # Prevent trivial self-loops
        if node["RawSupervisorId"] == pid:
            node["RawSupervisorId"] = None

    active_pids = {pid for pid, n in nodes.items() if n["Leaver"] == 0}
    if root_id not in active_pids and root_id in nodes:
        nodes[root_id]["Leaver"] = 0
        active_pids.add(root_id)

    # Step A: Walk upward from each active node to find its nearest active ancestor
    # (bridging across Leaver == 1 nodes if needed)
    active_parent = {}
    initial_status = {}

    for pid in active_pids:
        if pid == root_id:
            active_parent[pid] = None
            initial_status[pid] = "Root"
            continue

        curr = nodes[pid]["RawSupervisorId"]
        visited = {pid}
        steps = 0
        found_parent = None
        hit_cycle = False

        while curr is not None and steps < 50:
            if curr in visited:
                hit_cycle = True
                break
            visited.add(curr)
            steps += 1

            target_node = nodes.get(curr)
            if target_node is None:
                # SupervisorId does not exist in vw_StaffHierarchy at all
                break
            if target_node["Leaver"] == 0:
                found_parent = curr
                break
            # Otherwise target_node is a Leaver (Leaver == 1); bridge through to their supervisor
            curr = target_node["RawSupervisorId"]

        active_parent[pid] = found_parent
        if hit_cycle:
            initial_status[pid] = "Cycle Repaired"
        elif found_parent is not None:
            if steps > 1:
                initial_status[pid] = "Bridged via Leaver"
            elif pid in overridden_pids:
                initial_status[pid] = "Manual Override"
            else:
                initial_status[pid] = "Direct"
        else:
            initial_status[pid] = "Orphan"

    # Step B: Detect & break any cycles in the active-only graph (active_parent)
    state = {}  # 0=unvisited, 1=visiting, 2=visited
    for pid in sorted(active_pids):
        if state.get(pid, 0) != 0:
            continue
        trace = []
        curr = pid
        while curr is not None and state.get(curr, 0) == 0:
            state[curr] = 1
            trace.append(curr)
            curr = active_parent.get(curr)

        if curr is not None and state.get(curr, 0) == 1:
            active_parent[curr] = None
            initial_status[curr] = "Cycle Repaired"

        for u in trace:
            state[u] = 2

    # Step C: Build children adjacency map in the active graph
    children_map = defaultdict(list)
    for pid, par in active_parent.items():
        if par is not None and par in active_pids:
            children_map[par].append(pid)

    memo_subtree = {}
    def get_subtree_size(u):
        if u in memo_subtree:
            return memo_subtree[u]
        total = 0
        for ch in children_map.get(u, []):
            total += 1 + get_subtree_size(ch)
        memo_subtree[u] = total
        return total

    for pid in active_pids:
        get_subtree_size(pid)

    # Identify the Main Connected Tree rooted at root_id via BFS
    main_tree_depth = {}
    if root_id in active_pids:
        q = deque([(root_id, 1)])
        main_tree_depth[root_id] = 1
        while q:
            u, d = q.popleft()
            for ch in children_map.get(u, []):
                if ch not in main_tree_depth:
                    main_tree_depth[ch] = d + 1
                    q.append((ch, d + 1))

    # Step D: Find all disconnected subtree roots
    disconnected_roots = [
        pid for pid in active_pids
        if pid != root_id and pid not in main_tree_depth and active_parent.get(pid) is None
    ]

    disconnected_roots.sort(
        key=lambda x: (
            -memo_subtree.get(x, 0),
            0 if nodes[x]["IsSupervisor"] else 1,
            int(x) if x.isdigit() else 999999999,
            x,
        )
    )

    def pick_best_lead(candidates):
        if not candidates:
            return None
        return min(
            candidates,
            key=lambda c: (
                main_tree_depth.get(c, 999),
                -memo_subtree.get(c, 0),
                int(c) if c.isdigit() else 999999999,
                c,
            ),
        )

    for r_pid in disconnected_roots:
        r_node = nodes[r_pid]
        chosen_lead = None
        fallback_reason = None

        def find_unit_lead(field_name):
            val = r_node[field_name]
            if not val or val.lower() == "unknown":
                return None
            sup_pool = [
                c for c in main_tree_depth
                if c != root_id
                and nodes[c][field_name] == val
                and (nodes[c]["IsSupervisor"] or len(children_map.get(c, [])) > 0)
            ]
            return pick_best_lead(sup_pool)

        # Tier 1: Level6 Lead
        chosen_lead = find_unit_lead("Level6")
        if chosen_lead:
            fallback_reason = "Fallback: Level6 Lead"

        # Tier 2: CostCentre Lead
        if not chosen_lead:
            chosen_lead = find_unit_lead("CostCentre")
            if chosen_lead:
                fallback_reason = "Fallback: CostCentre Lead"

        # Tier 3: Level3 (Directorate) Lead
        if not chosen_lead:
            chosen_lead = find_unit_lead("Level3")
            if chosen_lead:
                fallback_reason = "Fallback: Level3 Lead"

        # Tier 4: Root (4037)
        if not chosen_lead:
            chosen_lead = root_id if root_id in active_pids else None
            fallback_reason = "Fallback: Root (4037)"

        if chosen_lead:
            active_parent[r_pid] = chosen_lead
            children_map[chosen_lead].append(r_pid)
            if initial_status.get(r_pid) != "Cycle Repaired":
                initial_status[r_pid] = fallback_reason

            base_depth = main_tree_depth.get(chosen_lead, 1)
            q = deque([(r_pid, base_depth + 1)])
            main_tree_depth[r_pid] = base_depth + 1
            while q:
                u, d = q.popleft()
                for ch in children_map.get(u, []):
                    if ch not in main_tree_depth:
                        main_tree_depth[ch] = d + 1
                        q.append((ch, d + 1))

            anc = chosen_lead
            added_size = 1 + memo_subtree.get(r_pid, 0)
            visited_anc = set()
            while anc is not None and anc not in visited_anc:
                visited_anc.add(anc)
                memo_subtree[anc] = memo_subtree.get(anc, 0) + added_size
                anc = active_parent.get(anc)

    # Step E: Final tree pass from Root to compute exact OrgPath, HierarchyDepth,
    # DirectReportsCount, and TotalReportsCount
    final_children = defaultdict(list)
    for pid in active_pids:
        par = active_parent.get(pid)
        if par is not None and par in active_pids:
            final_children[par].append(pid)

    final_total_reports = {}
    def compute_final_totals(u, visited=None):
        if visited is None:
            visited = set()
        if u in final_total_reports:
            return final_total_reports[u]
        visited.add(u)
        tot = 0
        for ch in final_children.get(u, []):
            if ch not in visited:
                tot += 1 + compute_final_totals(ch, visited)
        final_total_reports[u] = tot
        return tot

    for pid in active_pids:
        compute_final_totals(pid)

    resolved_records = []
    for pid in sorted(active_pids):
        chain = [pid]
        curr = active_parent.get(pid)
        seen = {pid}
        while curr is not None and curr not in seen:
            seen.add(curr)
            chain.append(curr)
            curr = active_parent.get(curr)
        chain.reverse()
        org_path = "|".join(chain)
        depth = len(chain)
        direct_cnt = len(final_children.get(pid, []))
        total_cnt = final_total_reports.get(pid, 0)

        resolved_records.append({
            "PersonFk": pid,
            "SupervisorId": active_parent.get(pid),
            "OriginalSupervisorId": nodes[pid]["OriginalSupervisorId"],
            "OrgPath": org_path,
            "HierarchyStatus": initial_status.get(pid, "Direct"),
            "HierarchyDepth": depth,
            "DirectReportsCount": direct_cnt,
            "TotalReportsCount": total_cnt,
        })

    return resolved_records

# ==============================================================================
# 3. EXECUTE ON LAKEHOUSE TABLE & VALIDATE INTEGRITY
# ==============================================================================
df_raw = spark.read.table(SOURCE_TABLE)

# Dynamically select only columns present in support_rls_dynamic_merged
# (Note: IsSupervisor is not in support_rls_dynamic_merged; supervisor status is derived from the graph)
available_cols = [
    c for c in ["PersonFk", "SupervisorId", "Leaver", "Level3", "CostCentre", "Level6"]
    if c in df_raw.columns
]
input_rows = [row.asDict() for row in df_raw.select(*available_cols).collect()]

resolved_list = resolve_staff_hierarchy(
    input_rows,
    root_id=ROOT_PERSON_FK,
    manual_overrides=MANUAL_SUPERVISOR_OVERRIDES,
)

# Validation Safeguards: Verify single connected tree before writing to Gold
total_active = len(resolved_list)
roots_found = [r["PersonFk"] for r in resolved_list if r["SupervisorId"] is None]
disconnected_paths = [r["PersonFk"] for r in resolved_list if not r["OrgPath"].startswith(ROOT_PERSON_FK)]

print(f"Total active staff resolved: {total_active}")
print(f"Root nodes count: {len(roots_found)} -> {roots_found}")
print(f"Disconnected paths not starting with {ROOT_PERSON_FK}: {len(disconnected_paths)}")

status_counts = defaultdict(int)
for r in resolved_list:
    status_counts[r["HierarchyStatus"]] += 1
print("Hierarchy Status Breakdown:")
for st, cnt in sorted(status_counts.items(), key=lambda x: -x[1]):
    print(f"  - {st}: {cnt}")

assert len(roots_found) <= 1, f"SAFEGUARD FAILED: Multiple root nodes detected: {roots_found}"
assert len(disconnected_paths) == 0, f"SAFEGUARD FAILED: {len(disconnected_paths)} nodes do not connect to root {ROOT_PERSON_FK}"

# Convert resolved graph metadata back to Spark DataFrame
resolved_schema = T.StructType([
    T.StructField("PersonFk", T.StringType(), False),
    T.StructField("SupervisorId", T.StringType(), True),
    T.StructField("OriginalSupervisorId", T.StringType(), True),
    T.StructField("OrgPath", T.StringType(), False),
    T.StructField("HierarchyStatus", T.StringType(), False),
    T.StructField("HierarchyDepth", T.IntegerType(), False),
    T.StructField("DirectReportsCount", T.IntegerType(), False),
    T.StructField("TotalReportsCount", T.IntegerType(), False),
])

df_resolved = spark.createDataFrame(resolved_list, schema=resolved_schema)

# Deduplicate df_raw by PersonFk (preferring Leaver == 0) and drop raw SupervisorId so our cleaned SupervisorId replaces it
df_raw_clean = (
    df_raw
    .withColumn("PersonFk", F.trim(F.col("PersonFk").cast("string")))
    .orderBy(F.col("Leaver").asc_nulls_last())
    .dropDuplicates(["PersonFk"])
    .drop("SupervisorId")
)

final_df = df_resolved.join(df_raw_clean, on="PersonFk", how="inner")

# Write to Gold Delta Table
(
    final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)
print(f"Successfully wrote {total_active} rows to {TARGET_TABLE}.")
